In [1]:
!pip install -q ollama
!pip install -q requests


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
OLLAMA_API_KEY = "8cf0f77083c64fd2b7d074bcc8e06092.SE8yHSR661XsLh7xeUFqI1sn"

In [3]:
import json
import math
import requests
from ollama import Client


HOST = 'https://ollama.com'
HEADERS = {'Authorization': f'Bearer {OLLAMA_API_KEY}'}

cliente = Client(host=HOST, headers=HEADERS)

# Modelo com suporte a ferramentas. Confira se está disponível no seu plano.
MODELO = 'gpt-oss:20b'
MAX_RODADAS = 6

In [4]:
CATALOGO = {'camiseta': 35, 'caneca': 22}
FRETES = {'centro': 8, 'alvorada': 12}


def consultar_produto(item):
    nome = str(item).strip().lower()
    preco = CATALOGO.get(nome)
    if preco is None:
        return {'item': nome, 'encontrado': False,
                'disponiveis': sorted(CATALOGO)}
    return {'item': nome, 'encontrado': True, 'preco_unitario_reais': preco}


def consultar_frete(bairro):
    nome = str(bairro).strip().lower()
    frete = FRETES.get(nome)
    if frete is None:
        return {'bairro': nome, 'encontrado': False,
                'atendidos': sorted(FRETES)}
    return {'bairro': nome, 'encontrado': True, 'frete_por_pedido_reais': frete}


def _numero(valor, nome, minimo, maximo):
    # O modelo às vezes manda números como texto: '2' -> 2.0
    if isinstance(valor, bool):
        raise ValueError(f'{nome} deve ser número')
    try:
        valor = float(valor)
    except (TypeError, ValueError):
        raise ValueError(f'{nome} deve ser número')
    if not math.isfinite(valor):
        raise ValueError(f'{nome} deve ser finito')
    if not (minimo <= valor <= maximo):
        raise ValueError(f'{nome} deve estar entre {minimo} e {maximo}')
    return valor


def calcular_total(quantidade, preco, frete):
    quantidade = _numero(quantidade, 'quantidade', 1, 1000)
    if not quantidade.is_integer():
        raise ValueError('quantidade deve ser inteira')
    preco = _numero(preco, 'preco', 0, 100_000)
    frete = _numero(frete, 'frete', 0, 10_000)
    total = round(int(quantidade) * preco + frete, 2)
    return {'quantidade': int(quantidade), 'preco_unitario_reais': preco,
            'frete_reais': frete, 'total_reais': total}

In [5]:
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'consultar_produto',
            'description': 'Consulta o catálogo da loja. Use para obter o preço '
                           'unitário de um item, em reais. Devolve encontrado=false '
                           'se o item não existir.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'item': {'type': 'string',
                             'description': 'Nome do produto no singular, ex.: camiseta'}
                },
                'required': ['item'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'consultar_frete',
            'description': 'Consulta o frete de entrega para um bairro. O valor é '
                           'em reais e cobrado UMA vez por pedido, não por unidade. '
                           'Devolve encontrado=false se o bairro não for atendido.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'bairro': {'type': 'string', 'description': 'Nome do bairro, ex.: Centro'}
                },
                'required': ['bairro'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calcular_total',
            'description': 'Calcula o total do orçamento: quantidade * preco + frete. '
                           'Use SOMENTE depois de obter preco e frete pelas consultas.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'quantidade': {'type': 'integer', 'description': 'Número de unidades (>= 1)'},
                    'preco': {'type': 'number', 'description': 'Preço unitário em reais, vindo de consultar_produto'},
                    'frete': {'type': 'number', 'description': 'Frete do pedido em reais, vindo de consultar_frete'},
                },
                'required': ['quantidade', 'preco', 'frete'],
            },
        },
    },
]

SYSTEM = (
    'Você é o atendente de uma loja. Responda em português, de forma curta. '
    'Nunca invente preços, fretes ou totais: use sempre as ferramentas. '
    'Consulte preço e frete antes de calcular o total. '
    'Se um item ou bairro não for encontrado, diga isso ao cliente e mostre as opções disponíveis. '
    'O frete é cobrado uma vez por pedido.'
)

In [6]:
FERRAMENTAS = {
    'consultar_produto': consultar_produto,
    'consultar_frete': consultar_frete,
    'calcular_total': calcular_total,
}


def executar_ferramenta(nome, argumentos):
    if nome not in FERRAMENTAS:
        return {'erro': f'ferramenta desconhecida: {nome}'}
    if not isinstance(argumentos, dict):
        return {'erro': 'argumentos devem ser um objeto JSON'}
    try:
        return FERRAMENTAS[nome](**argumentos)
    except TypeError as e:          # argumento faltando ou sobrando
        return {'erro': f'argumentos inválidos: {e}'}
    except ValueError as e:         # validação de calcular_total
        return {'erro': str(e)}



In [7]:
def _para_nativo(msg):
    if msg['role'] == 'assistant' and msg.get('tool_calls'):
        return {'role': 'assistant', 'content': msg.get('content') or '',
                'tool_calls': [{'function': {'name': c['name'], 'arguments': c['arguments']}}
                               for c in msg['tool_calls']]}
    if msg['role'] == 'tool':
        return {'role': 'tool', 'tool_name': msg['tool_name'], 'content': msg['content']}
    return {'role': msg['role'], 'content': msg.get('content') or ''}


def pedir_nativo(historico, modelo):
    response = cliente.chat(
        model=modelo,
        messages=[_para_nativo(m) for m in historico],
        tools=TOOLS,
        options={'num_predict': 1024},
    )
    m = response.message
    chamadas = [
        {'id': f'nativo_{i}', 'name': c.function.name, 'arguments': dict(c.function.arguments or {})}
        for i, c in enumerate(m.tool_calls or [])
    ]
    return {'role': 'assistant', 'content': m.content or '', 'tool_calls': chamadas}

In [8]:
def _para_compativel(msg):
    if msg['role'] == 'assistant' and msg.get('tool_calls'):
        return {'role': 'assistant', 'content': msg.get('content') or '',
                'tool_calls': [{'id': c['id'], 'type': 'function',
                                'function': {'name': c['name'],
                                             'arguments': json.dumps(c['arguments'])}}
                               for c in msg['tool_calls']]}
    if msg['role'] == 'tool':
        return {'role': 'tool', 'tool_call_id': msg['tool_call_id'], 'content': msg['content']}
    return {'role': msg['role'], 'content': msg.get('content') or ''}


def pedir_compativel(historico, modelo):
    r = requests.post(
        f'{HOST}/v1/chat/completions',
        headers=HEADERS,
        json={'model': modelo,
              'messages': [_para_compativel(m) for m in historico],
              'tools': TOOLS,
              'max_tokens': 1024},
        timeout=120,
    )
    r.raise_for_status()
    m = r.json()['choices'][0]['message']
    chamadas = []
    for c in m.get('tool_calls') or []:
        try:
            args = json.loads(c['function'].get('arguments') or '{}')
        except json.JSONDecodeError:
            args = None   # executar_ferramenta devolve erro ao modelo
        chamadas.append({'id': c['id'], 'name': c['function']['name'], 'arguments': args})
    return {'role': 'assistant', 'content': m.get('content') or '', 'tool_calls': chamadas}

In [9]:
def rodar_agente(historico, pedir, modelo, max_rodadas=MAX_RODADAS, verbose=True):
    for rodada in range(1, max_rodadas + 1):
        mensagem = pedir(historico, modelo)
        historico.append(mensagem)                     # 1º a solicitação

        if not mensagem['tool_calls']:                 # sem chamadas: resposta final
            return mensagem['content']

        for chamada in mensagem['tool_calls']:         # execução sequencial
            resultado = executar_ferramenta(chamada['name'], chamada['arguments'])
            if verbose:
                print(f'  [rodada {rodada}] {chamada["name"]}({chamada["arguments"]}) -> {resultado}')
            historico.append({                         # 2º o resultado
                'role': 'tool',
                'tool_call_id': chamada['id'],
                'tool_name': chamada['name'],
                'content': json.dumps(resultado, ensure_ascii=False),
            })

    return f'(parei: limite de {max_rodadas} rodadas atingido)'


def novo_historico():
    return [{'role': 'system', 'content': SYSTEM}]

---
# Desafios

## Desafio 1 — Receba o pedido com `input()`

Teste outro item, outra quantidade e outro bairro — e confira uma conta no Python.

In [10]:
pedido = input('Seu pedido: ')
historico = novo_historico()
historico.append({'role': 'user', 'content': pedido})
print('\nResposta:', rodar_agente(historico, pedir_nativo, MODELO))

# Conferência manual, ex.: 3 canecas para a Alvorada
print('Conferência:', calcular_total(3, CATALOGO['caneca'], FRETES['alvorada']))


Resposta: Aqui estão os produtos que temos disponíveis atualmente:

- Camiseta
- Calça
- Blusa
- Tênis
- Chapéu  

Se precisar de alguma informação adicional (preço, frete ou detalhes), é só falar!
Conferência: {'quantidade': 3, 'preco_unitario_reais': 22.0, 'frete_reais': 12.0, 'total_reais': 78.0}


## Desafio 2 — Acrescente uma ferramenta

Exemplo: `verificar_estoque`. Três passos:
1. criar a função
2. **registrar** o nome em `FERRAMENTAS`
3. **descrever** em `TOOLS` (e, se quiser, citar nas regras do `SYSTEM`)

`rodar_agente` e os adaptadores não mudam.

In [11]:
ESTOQUE = {'camiseta': 10, 'caneca': 0}


def verificar_estoque(item, quantidade):
    nome = str(item).strip().lower()
    if nome not in ESTOQUE:
        return {'item': nome, 'encontrado': False}
    disponivel = ESTOQUE[nome]
    return {'item': nome, 'encontrado': True, 'disponivel': disponivel,
            'atende': disponivel >= int(quantidade)}


FERRAMENTAS['verificar_estoque'] = verificar_estoque
if not any(t['function']['name'] == 'verificar_estoque' for t in TOOLS):
    TOOLS.append({
        'type': 'function',
        'function': {
            'name': 'verificar_estoque',
            'description': 'Verifica se há unidades suficientes de um item em estoque. '
                           'Use antes de confirmar um orçamento.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'item': {'type': 'string', 'description': 'Nome do produto'},
                    'quantidade': {'type': 'integer', 'description': 'Unidades desejadas'},
                },
                'required': ['item', 'quantidade'],
            },
        },
    })

SYSTEM += ' Antes de fechar o orçamento, verifique o estoque; se não houver unidades suficientes, avise o cliente.'

historico = novo_historico()
historico.append({'role': 'user', 'content': 'Quero 2 canecas para o Centro. Quanto fica?'})
print('\nResposta:', rodar_agente(historico, pedir_nativo, MODELO))

  [rodada 1] consultar_produto({'item': 'caneca'}) -> {'item': 'caneca', 'encontrado': True, 'preco_unitario_reais': 22}
  [rodada 2] consultar_frete({'bairro': 'Centro'}) -> {'bairro': 'centro', 'encontrado': True, 'frete_por_pedido_reais': 8}
  [rodada 3] calcular_total({'frete': 8, 'preco': 22, 'quantidade': 2}) -> {'quantidade': 2, 'preco_unitario_reais': 22.0, 'frete_reais': 8.0, 'total_reais': 52.0}
  [rodada 4] verificar_estoque({'item': 'caneca', 'quantidade': 2}) -> {'item': 'caneca', 'encontrado': True, 'disponivel': 0, 'atende': False}

Resposta: Desculpe, mas não temos quantidade disponível para 2 canecas no estoque atual. Se desejar, posso propor a compra de 1 unidade ou pode indicar outro produto que esteja disponível.


## Desafio 3 — Deixe a conversa continuar

Dois laços com funções diferentes:
- **laço das perguntas** (`while` + `input`): uma volta por mensagem da pessoa
- **laço das ferramentas** (`rodar_agente`): várias rodadas até a resposta final

O histórico **não é recriado** entre perguntas — é ele que permite entender “e se forem 3?”.
Digite `novo` para recomeçar com histórico vazio e `sair` para encerrar.

Sugestão de teste:
1. `Quero 2 camisetas no Centro.`
2. `E se forem 3?` → reaproveita item e bairro
3. `Agora manda para a Alvorada.` → só o frete muda: nova consulta de frete e novo cálculo
4. `novo` e depois `E se forem 3?` → sem memória, o modelo não sabe do que se trata

In [12]:
historico = novo_historico()
print("Atendente pronto. Digite 'sair' para encerrar ou 'novo' para recomeçar.")

while True:
    texto = input('\nVocê: ').strip()
    if texto.lower() == 'sair':
        break
    if texto.lower() == 'novo':
        historico = novo_historico()
        print('(histórico reiniciado)')
        continue
    if not texto:
        continue
    historico.append({'role': 'user', 'content': texto})
    print('Atendente:', rodar_agente(historico, pedir_nativo, MODELO))

print(f'\nFim. O histórico tinha {len(historico)} mensagens.')

Atendente pronto. Digite 'sair' para encerrar ou 'novo' para recomeçar.
Atendente: Qual é o seu bairro para que eu possa verificar a disponibilidade e o frete?
  [rodada 1] consultar_frete({'bairro': 'Sao Lazaro'}) -> {'bairro': 'sao lazaro', 'encontrado': False, 'atendidos': ['alvorada', 'centro']}
Atendente: Desculpe, não atendemos o bairro São Lázaro.  
Os bairros atendidos são: "alvorada" e "centro". Por favor, informe de qual deles você quer!
  [rodada 1] consultar_produto({'item': 'camisa'}) -> {'item': 'camisa', 'encontrado': False, 'disponiveis': ['camiseta', 'caneca']}
Atendente: Desculpe, não há "camisa" no catálogo.  
Itens disponíveis: camiseta, caneca. Qual deles você deseja?
Atendente: Peço desculpas, não tem “camisa” no nosso estoque.  
Você pode escolher entre:

* Camiseta – R$ __ (preciso consultar preço)  
* Caneca – R$ __ (preciso consultar preço)

Qual dessas opções você gostaria de experimentar?
  [rodada 1] consultar_produto({'item': 'camiseta'}) -> {'item': 'cami

## Desafio 4 — Troque o modelo

Mantenha pedido, funções e regras. Troque só o nome do modelo e compare:
total, ferramentas usadas, dados ausentes e número de rodadas.
**Histórico novo em cada tentativa.** Use modelos com suporte a ferramentas e disponíveis no seu plano.

In [ ]:
PEDIDO = 'Quanto custam 2 camisetas para o Centro?'
MODELOS = ['gpt-oss:20b', 'gpt-oss:120b']   # ajuste para os modelos do seu plano


def comparar(pedir, modelo):
    historico = novo_historico()
    historico.append({'role': 'user', 'content': PEDIDO})
    try:
        resposta = rodar_agente(historico, pedir, modelo, verbose=False)
    except Exception as e:
        return {'modelo': modelo, 'erro': str(e)[:120]}
    usadas = [c['name'] for m in historico if m['role'] == 'assistant' for c in m['tool_calls']]
    rodadas = sum(1 for m in historico if m['role'] == 'assistant')
    return {'modelo': modelo, 'rodadas': rodadas, 'ferramentas': usadas,
            'tem_78': '78' in resposta, 'resposta': resposta[:120]}


for modelo in MODELOS:
    print(comparar(pedir_nativo, modelo))

## Desafio 5 — Troque somente o backend

`pedir_compativel` fala com `/v1/chat/completions`. O laço `rodar_agente` **não muda**.

In [ ]:
backend = pedir_compativel

# Comece com um histórico novo.
historico = novo_historico()
historico.append({'role': 'user', 'content': PEDIDO})
resposta = rodar_agente(historico, backend, MODELO)
print('\nResposta:', resposta)

In [ ]:
# Lado a lado: mesmo pedido, mesmo modelo, dois backends
for b in (pedir_nativo, pedir_compativel):
    print(b.__name__, '->', comparar(b, MODELO))